# Engenharia de Contexto em RAG — versão didática

Este notebook deriva do `genai.ipynb` fornecido pela professora e adiciona duas
camadas de apoio:

1. explicações dos conceitos apresentados em `aula_2.pdf`;
2. um roteiro para coletar as evidências solicitadas em `Atividade.pdf`.

O algoritmo-base foi preservado. As células marcadas com a tag
`codigo-original` contêm literalmente o código da professora, apenas dividido
pelos títulos que já existiam no arquivo. A chamada interativa `main()` foi
movida para uma célula opcional no final, porque ela abre um laço de perguntas e
interromperia o roteiro guiado.

> O notebook orienta a execução, mas não escolhe a pergunta, não preenche a
> análise e não conclui qual configuração é melhor. Essas decisões fazem parte
> da atividade.


## O que precisa aparecer na entrega

O relatório deve acompanhar uma mesma pergunta em três configurações:

| Etapa | O que executar | O que registrar |
|---|---|---|
| Baseline | Busca por similaridade | Top-3, escopo, tokens, latência e suficiência do top-1 |
| Avançado | Pipeline completo | Consultas geradas, candidatos, filtros, deduplicação, reranking, tokens e decisão |
| Alterado | Uma modificação controlada | A mesma medição para permitir comparação justa |

Ao final, compare relevância, ruído, redundância, adequação do escopo, custo em
tokens, latência e suficiência. Uma métrica isolada não prova que o contexto
melhorou: reduzir tokens, por exemplo, pode remover uma condição importante.


## 1. Ambiente

Use o kernel **Python (aula04)**. A instalação abaixo lê o arquivo
`requirements.txt` do workspace. Normalmente ela só confirma pacotes que já
estão instalados no ambiente virtual.


In [ ]:
%pip install -r requirements.txt

### Diretório de trabalho

Os caminhos do notebook são relativos. Portanto, o Jupyter deve estar aberto
na pasta `CONT/aula04`. A célula seguinte mostra o diretório efetivamente usado.


In [1]:
from pathlib import Path

print(f"Workspace local: {Path.cwd().resolve()}")


Workspace local: /home/ubuntu/mba-genai/CONT/aula04


### Cache local do modelo de embeddings

O modelo de embeddings já foi baixado para este ambiente. A próxima célula
impede consultas desnecessárias à internet durante o carregamento; ela não
altera o modelo nem o algoritmo de recuperação. Se o notebook for aberto em
outro computador sem o modelo em cache, defina `HF_HUB_OFFLINE = "0"` uma vez
para permitir o download inicial.


In [2]:
import os

# Reutiliza o modelo já armazenado em ~/.cache/huggingface.
# Isso evita que um novo kernel fique aguardando uma consulta de rede.
os.environ["HF_HUB_OFFLINE"] = "1"

print("Modelo de embeddings: uso do cache local ativado.")


Modelo de embeddings: uso do cache local ativado.


### Documentos usados na recuperação

A pasta `documentos/` contém três fontes com papéis diferentes:

- regulamento de estágio → escopo `estagio`;
- PPC do curso → escopo `curso`;
- ficha de Banco de Dados (CI1218) → escopo `disciplina`.

Essa distinção é central para perguntas ambíguas. “Frequência mínima”, por
exemplo, pode significar regras diferentes dependendo da fonte consultada.


In [3]:
from pathlib import Path
from time import perf_counter
from dataclasses import dataclass
import hashlib
import math
import re
import unicodedata

import chromadb
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


# ============================================================
# CONFIGURAÇÕES
# ============================================================

PASTA_DOCUMENTOS = Path("documentos")
PASTA_CHROMA = Path("chroma_db")


/home/ubuntu/mba-genai/CONT/aula04/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Parâmetros do experimento

Estes valores controlam o comportamento observado na atividade:

- `TAMANHO_CHUNK` e `OVERLAP_CHUNK`: quantidade de caracteres por trecho e
  repetição entre trechos vizinhos;
- `TOP_K`: quantidade final de resultados;
- `MODO_BUSCA`: similaridade pura ou MMR no fluxo baseline interativo;
- `FETCH_K_MMR` e `LAMBDA_MMR`: candidatos e equilíbrio entre relevância e
  diversidade no MMR;
- `RECRIAR_COLECAO`: recria ou reutiliza o índice persistente;
- `LIMITE_TOKENS_CONTEXTO`: orçamento aproximado enviado ao LLM;
- `LIMIAR_SIMILARIDADE`: remove candidatos abaixo do score definido.

Na primeira indexação, mantenha `RECRIAR_COLECAO = True`. Depois, use `False`
para evitar reconstruir a coleção a cada execução. Para o experimento alterado,
mude uma variável por vez; isso permite atribuir o efeito à decisão correta.


In [4]:
NOME_COLECAO = "documentos_academicos_aula"

MODELO_EMBEDDING = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

TAMANHO_CHUNK = 800
OVERLAP_CHUNK = 150
TAMANHO_LOTE = 32
TOP_K = 3

# Modos disponíveis: "similaridade" ou "mmr".
MODO_BUSCA = "similaridade"

# No MMR, primeiro recuperamos um conjunto maior de candidatos.
FETCH_K_MMR = 12

# Próximo de 1: prioriza relevância.
# Menor: aumenta a diversidade.
LAMBDA_MMR = 0.6

# Na primeira execução desta versão, deixe True para recriar
# a coleção com os metadados de escopo e tipo de documento.
RECRIAR_COLECAO = False

# "baseline" mantém o comportamento da aula anterior.
# "avancado" executa o pipeline de engenharia de contexto.
MODO_PIPELINE = "avancado"

# Escopo opcional: "curso", "disciplina", "estagio" ou None.
# None é útil para verificar perguntas ambíguas.
ESCOPO_PADRAO = None

# Orçamento aproximado do contexto enviado ao LLM.
LIMITE_TOKENS_CONTEXTO = 700

# Limiar didático para eliminar candidatos fracos.
LIMIAR_SIMILARIDADE = 0.20

ESCOPOS_VALIDOS = {"curso", "disciplina", "estagio"}




## 3. Normalização

As funções abaixo removem diferenças superficiais — maiúsculas, acentos,
espaços e quebras de linha — antes de comparar termos. Isso não produz
embeddings e não muda o significado semântico; apenas torna regras como
“estágio” versus “estagio” previsíveis.


In [5]:
# ============================================================
# UTILITÁRIOS DE NORMALIZAÇÃO
# ============================================================

def remover_acentos(texto: str) -> str:
    decomposed = unicodedata.normalize("NFD", texto)
    return "".join(
        char
        for char in decomposed
        if unicodedata.category(char) != "Mn"
    )


def normalizar_para_busca(texto: str) -> str:
    texto = remover_acentos(texto.lower())
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return " ".join(texto.split())


def normalizar_texto(texto: str) -> str:
    texto = texto.replace("\u00a0", " ")
    texto = texto.replace("\r\n", "\n")
    texto = texto.replace("\r", "\n")
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n\s*\n+", "\n\n", texto)
    return texto.strip()


def normalizar_trecho_contexto(texto: str) -> str:
    """Normalização leve do contexto que será enviado ao LLM."""
    texto = texto.replace("\n", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def validar_escopo(escopo: str | None) -> None:
    if escopo is None:
        return

    if escopo not in ESCOPOS_VALIDOS:
        raise ValueError(
            f"Escopo inválido: {escopo}. "
            f"Use um de: {sorted(ESCOPOS_VALIDOS)}"
        )




## 4. Ingestão, escopo e metadados

Cada PDF é lido página a página. O texto extraído recebe dois metadados:
`escopo` e `tipo_documento`. O código prioriza o nome do arquivo, uma fonte
controlada, e usa o conteúdo inicial da página apenas como fallback.

Esse desenho combina recuperação semântica com regras simbólicas. O embedding
indica proximidade de significado; o metadado impede que uma regra de estágio
seja tratada como regra de disciplina apenas porque as frases são parecidas.


In [6]:
# ============================================================
# INGESTÃO: LEITURA, ESCOPO E METADADOS
# ============================================================

def inferir_escopo(arquivo: str, texto: str = "") -> str:
    """
    Inferência didática de escopo.

    A decisão prioriza o nome do arquivo, que funciona como uma fonte
    simbólica/controlada. O texto é usado apenas como fallback.
    """
    nome = normalizar_para_busca(arquivo)

    # Primeiro: decisões controladas pelo nome/fonte do documento.
    if "estagio" in nome:
        return "estagio"

    if "ci1218" in nome or "disciplina" in nome or "ficha" in nome:
        return "disciplina"

    if "ppc" in nome or "projeto pedagogico" in nome:
        return "curso"

    # Fallback: conteúdo da página.
    conteudo = normalizar_para_busca(texto[:1200])

    if "ficha de disciplina" in conteudo or "ementa" in conteudo:
        return "disciplina"

    if "regulamento de estagio" in conteudo:
        return "estagio"

    if "projeto pedagogico" in conteudo or "ppc" in conteudo:
        return "curso"

    return "geral"


def inferir_tipo_documento(arquivo: str, texto: str = "") -> str:
    nome = normalizar_para_busca(arquivo)
    conteudo = normalizar_para_busca(texto[:1200])
    alvo = f"{nome} {conteudo}"

    if "regulamento" in alvo:
        return "regulamento"
    if "ficha" in alvo or "ementa" in alvo:
        return "ficha_disciplina"
    if "ppc" in alvo or "projeto pedagogico" in alvo:
        return "ppc"
    return "documento"


def carregar_pdfs(pasta: Path) -> list[dict]:
    if not pasta.exists():
        raise FileNotFoundError(
            f"A pasta não existe: {pasta.resolve()}"
        )

    arquivos = sorted(pasta.glob("*.pdf"))

    if not arquivos:
        raise FileNotFoundError(
            f"Nenhum PDF encontrado em: {pasta.resolve()}"
        )

    paginas = []

    for arquivo in arquivos:
        print(f"Lendo: {arquivo.name}")
        reader = PdfReader(str(arquivo))

        for numero_pagina, pagina in enumerate(reader.pages, start=1):
            texto = normalizar_texto(pagina.extract_text() or "")

            if not texto:
                print(
                    f"  Aviso: página {numero_pagina} "
                    "sem texto extraível."
                )
                continue

            paginas.append(
                {
                    "arquivo": arquivo.name,
                    "pagina": numero_pagina,
                    "texto": texto,
                    "escopo": inferir_escopo(arquivo.name, texto),
                    "tipo_documento": inferir_tipo_documento(
                        arquivo.name,
                        texto,
                    ),
                }
            )

    return paginas




## 5. Chunking

O documento é dividido em janelas de caracteres com sobreposição. Chunks muito
grandes preservam mais contexto, mas gastam tokens e podem misturar assuntos;
chunks pequenos são específicos, porém podem separar uma regra de sua condição.
O overlap reduz cortes abruptos ao custo de aumentar redundância.


In [7]:
# ============================================================
# CHUNKING
# ============================================================

def gerar_chunks(
    texto: str,
    tamanho: int = TAMANHO_CHUNK,
    overlap: int = OVERLAP_CHUNK,
) -> list[str]:
    if tamanho <= 0:
        raise ValueError("O tamanho do chunk deve ser maior que zero.")

    if overlap < 0:
        raise ValueError("O overlap não pode ser negativo.")

    if overlap >= tamanho:
        raise ValueError("O overlap deve ser menor que o tamanho.")

    chunks = []
    inicio = 0
    passo = tamanho - overlap

    while inicio < len(texto):
        fim = min(inicio + tamanho, len(texto))
        chunk = texto[inicio:fim].strip()

        if chunk:
            chunks.append(chunk)

        inicio += passo

    return chunks


def preparar_chunks(paginas: list[dict]) -> list[dict]:
    todos_chunks = []

    for pagina in paginas:
        chunks_da_pagina = gerar_chunks(
            pagina["texto"],
            tamanho=TAMANHO_CHUNK,
            overlap=OVERLAP_CHUNK,
        )

        nome_base = Path(pagina["arquivo"]).stem

        for numero_chunk, texto_chunk in enumerate(
            chunks_da_pagina,
            start=1,
        ):
            identificador = (
                f"{nome_base}"
                f"_p{pagina['pagina']:03d}"
                f"_c{numero_chunk:03d}"
            )

            todos_chunks.append(
                {
                    "id": identificador,
                    "arquivo": pagina["arquivo"],
                    "pagina": pagina["pagina"],
                    "numero_chunk": numero_chunk,
                    "texto": texto_chunk,
                    "escopo": pagina["escopo"],
                    "tipo_documento": pagina["tipo_documento"],
                }
            )

    return todos_chunks




## 6. Banco vetorial

O ChromaDB persiste os vetores em `chroma_db/`. A coleção usa distância de
cosseno. Com embeddings normalizados, valores mais próximos representam maior
similaridade semântica. Recriar a coleção é útil quando documentos, chunks ou
metadados mudam; reutilizá-la reduz o custo das execuções seguintes.


In [8]:
# ============================================================
# BANCO VETORIAL
# ============================================================

def criar_colecao(recriar: bool):
    PASTA_CHROMA.mkdir(parents=True, exist_ok=True)

    client = chromadb.PersistentClient(path=str(PASTA_CHROMA))

    if recriar:
        try:
            client.delete_collection(NOME_COLECAO)
            print(f"Coleção anterior removida: {NOME_COLECAO}")
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name=NOME_COLECAO,
        configuration={"hnsw": {"space": "cosine"}},
    )

    return collection




## 7. Embeddings e indexação

`SentenceTransformer` transforma cada chunk em um vetor. A indexação é feita
em lotes e grava simultaneamente ID, vetor, texto e metadados. O modelo usado é
multilíngue, adequado aos documentos e perguntas em português.

O embedding ajuda a localizar texto semanticamente relacionado, mas não garante
que o trecho contenha a resposta. Essa diferença entre relevância e suficiência
será medida no exercício.


In [9]:
# ============================================================
# EMBEDDINGS E INDEXAÇÃO
# ============================================================

def indexar_chunks(
    collection,
    modelo: SentenceTransformer,
    chunks: list[dict],
) -> None:
    if not chunks:
        raise ValueError("Não existem chunks para indexar.")

    total = len(chunks)
    inicio_total = perf_counter()

    for inicio in range(0, total, TAMANHO_LOTE):
        fim = min(inicio + TAMANHO_LOTE, total)
        lote = chunks[inicio:fim]

        textos = [chunk["texto"] for chunk in lote]

        embeddings = modelo.encode(
            textos,
            batch_size=TAMANHO_LOTE,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        collection.upsert(
            ids=[chunk["id"] for chunk in lote],
            embeddings=embeddings.tolist(),
            documents=textos,
            metadatas=[
                {
                    "arquivo": chunk["arquivo"],
                    "pagina": chunk["pagina"],
                    "numero_chunk": chunk["numero_chunk"],
                    "tamanho_caracteres": len(chunk["texto"]),
                    "escopo": chunk["escopo"],
                    "tipo_documento": chunk["tipo_documento"],
                    "fonte_controlada": True,
                }
                for chunk in lote
            ],
        )

        print(f"Indexados: {fim}/{total}")

    tempo = perf_counter() - inicio_total

    print(f"Indexação concluída em {tempo:.2f} s.")
    print(f"Registros na coleção: {collection.count()}")




## 8. Baseline: busca por similaridade

O baseline codifica a pergunta, recupera os `k` vetores mais próximos e pode
aplicar um filtro de escopo. Ele serve como referência simples: qualquer ganho
do pipeline avançado deve ser comparado a este resultado, incluindo o custo
adicional.

Para cada execução, observe top-1, suficiência, escopo, redundância, ruído,
latência e tokens. “Relacionado” não significa necessariamente “capaz de
responder”.


In [10]:
# ============================================================
# BUSCA VETORIAL BASELINE
# ============================================================

def buscar(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    k: int = TOP_K,
    escopo: str | None = None,
) -> dict:
    pergunta = pergunta.strip()
    validar_escopo(escopo)

    if not pergunta:
        raise ValueError("A pergunta não pode estar vazia.")

    total_registros = collection.count()

    if total_registros == 0:
        raise RuntimeError("A coleção está vazia.")

    k_real = min(k, total_registros)

    embedding_pergunta = modelo.encode(
        pergunta,
        normalize_embeddings=True,
    ).tolist()

    inicio = perf_counter()

    parametros = {
        "query_embeddings": [embedding_pergunta],
        "n_results": k_real,
        "include": ["documents", "metadatas", "distances"],
    }

    # Filtro simbólico antes da busca vetorial.
    # Isso reduz o espaço de busca quando o escopo já é conhecido.
    if escopo is not None:
        parametros["where"] = {"escopo": escopo}

    resultados = collection.query(**parametros)

    resultados["latencia_ms"] = (perf_counter() - inicio) * 1000
    resultados["pergunta"] = pergunta
    resultados["metodo"] = "similaridade"
    resultados["escopo"] = escopo

    return resultados




## 9. MMR: relevância com diversidade

O Maximal Marginal Relevance seleciona resultados equilibrando dois objetivos:

$$\text{MMR}=\lambda\,\text{relevância}-(1-\lambda)\,\text{redundância}$$

Com `lambda` próximo de 1, a relevância domina. Valores menores favorecem
diversidade. MMR pode reduzir chunks repetidos, mas um resultado diferente não é
automaticamente melhor: ele ainda precisa conter evidência útil.

Importante para a atividade: `LAMBDA_MMR` afeta `buscar_mmr`. O pipeline
avançado usa múltiplas buscas por similaridade e RRF; portanto, alterar somente
`LAMBDA_MMR` não modifica `responder_com_contexto`.


In [11]:
# ============================================================
# MAXIMAL MARGINAL RELEVANCE
# ============================================================

def buscar_mmr(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    k: int = TOP_K,
    fetch_k: int = FETCH_K_MMR,
    lambda_mult: float = LAMBDA_MMR,
    escopo: str | None = None,
) -> dict:
    """
    Recupera fetch_k candidatos por similaridade e seleciona k
    resultados equilibrando relevância e diversidade.
    """
    pergunta = pergunta.strip()
    validar_escopo(escopo)

    if not pergunta:
        raise ValueError("A pergunta não pode estar vazia.")
    if k <= 0:
        raise ValueError("k deve ser maior que zero.")
    if fetch_k < k:
        raise ValueError("fetch_k deve ser maior ou igual a k.")
    if not 0 <= lambda_mult <= 1:
        raise ValueError("lambda_mult deve estar entre 0 e 1.")

    total_registros = collection.count()

    if total_registros == 0:
        raise RuntimeError("A coleção está vazia.")

    k_real = min(k, total_registros)
    fetch_k_real = min(max(fetch_k, k_real), total_registros)

    embedding_pergunta = modelo.encode(
        pergunta,
        normalize_embeddings=True,
    )

    inicio = perf_counter()

    parametros = {
        "query_embeddings": [embedding_pergunta.tolist()],
        "n_results": fetch_k_real,
        "include": [
            "documents",
            "metadatas",
            "distances",
            "embeddings",
        ],
    }

    if escopo is not None:
        parametros["where"] = {"escopo": escopo}

    candidatos = collection.query(**parametros)

    ids = candidatos["ids"][0]
    documentos = candidatos["documents"][0]
    metadados = candidatos["metadatas"][0]

    if not ids:
        return {
            "ids": [[]],
            "documents": [[]],
            "metadatas": [[]],
            "distances": [[]],
            "latencia_ms": (perf_counter() - inicio) * 1000,
            "pergunta": pergunta,
            "metodo": "MMR",
            "lambda_mmr": lambda_mult,
            "fetch_k": fetch_k_real,
            "escopo": escopo,
        }

    embeddings = np.asarray(
        candidatos["embeddings"][0],
        dtype=np.float32,
    )

    relevancias = embeddings @ embedding_pergunta
    selecionados = [int(np.argmax(relevancias))]

    while len(selecionados) < k_real and len(selecionados) < len(ids):
        melhor_indice = None
        melhor_score = float("-inf")

        for indice in range(len(ids)):
            if indice in selecionados:
                continue

            redundancias = embeddings[selecionados] @ embeddings[indice]
            maior_redundancia = float(np.max(redundancias))

            score_mmr = (
                lambda_mult * float(relevancias[indice])
                - (1 - lambda_mult) * maior_redundancia
            )

            if score_mmr > melhor_score:
                melhor_score = score_mmr
                melhor_indice = indice

        if melhor_indice is None:
            break

        selecionados.append(melhor_indice)

    latencia_ms = (perf_counter() - inicio) * 1000

    similaridades_selecionadas = [
        float(relevancias[indice])
        for indice in selecionados
    ]

    return {
        "ids": [[ids[indice] for indice in selecionados]],
        "documents": [[documentos[indice] for indice in selecionados]],
        "metadatas": [[metadados[indice] for indice in selecionados]],
        "distances": [[
            1 - similaridade
            for similaridade in similaridades_selecionadas
        ]],
        "latencia_ms": latencia_ms,
        "pergunta": pergunta,
        "metodo": "MMR",
        "lambda_mmr": lambda_mult,
        "fetch_k": fetch_k_real,
        "escopo": escopo,
    }




### Leitura dos resultados do baseline

A distância de cosseno é convertida em similaridade aproximada por
`1 - distancia`. Além do número, examine texto, arquivo, página e escopo. A
evidência correta precisa ser rastreável até uma fonte adequada.


In [12]:
# ============================================================
# EXIBIÇÃO DO BASELINE
# ============================================================

def imprimir_resultados(resultados: dict) -> None:
    documentos = resultados["documents"][0]
    metadados = resultados["metadatas"][0]
    distancias = resultados["distances"][0]
    ids = resultados["ids"][0]

    print("\n" + "#" * 80)
    print(f"Pergunta: {resultados['pergunta']}")
    print(f"Método: {resultados.get('metodo', 'similaridade')}")
    print(f"Escopo: {resultados.get('escopo')}")
    print(f"Latência da busca: {resultados['latencia_ms']:.2f} ms")
    print("#" * 80)

    for rank, (identificador, texto, metadata, distancia) in enumerate(
        zip(ids, documentos, metadados, distancias),
        start=1,
    ):
        similaridade = 1 - distancia

        print("\n" + "=" * 80)
        print(f"Rank: {rank}")
        print(f"ID: {identificador}")
        print(f"Arquivo: {metadata.get('arquivo')}")
        print(f"Página: {metadata.get('pagina')}")
        print(f"Chunk: {metadata.get('numero_chunk')}")
        print(f"Escopo: {metadata.get('escopo', 'nao_informado')}")
        print(f"Tipo: {metadata.get('tipo_documento', 'nao_informado')}")
        print(f"Distância de cosseno: {distancia:.4f}")
        print(f"Similaridade aproximada: {similaridade:.4f}")
        print("-" * 80)
        print(texto[:1200])




## 10. Engenharia de contexto: visão geral

O pipeline avançado não é apenas uma busca maior. Cada etapa responde a uma
pergunta diferente:

`ambiguidade → transformação → recuperação → fusão → filtros → deduplicação → reranking → compressão → suficiência → ação`

- recuperação encontra candidatos relacionados;
- filtros excluem candidatos incompatíveis;
- reranking altera a ordem;
- compressão escolhe o conteúdo que cabe no orçamento;
- suficiência decide se existe evidência para responder.

Misturar essas funções dificulta descobrir por que o resultado melhorou ou
piorou. O experimento deve registrar o efeito de cada decisão.


In [13]:
# ============================================================
# AULA 2: ENGENHARIA DE CONTEXTO
# ============================================================

@dataclass
class ResultadoChunk:
    id: str
    texto: str
    metadata: dict
    distancia: float
    similaridade: float
    consulta_origem: str


def estimar_tokens(texto: str) -> int:
    """
    Estimativa didática: 1 token ~= 4 caracteres.
    Para produção, use o tokenizer do modelo escolhido.
    """
    return math.ceil(len(texto) / 4)


def estimar_tokens_resultados(resultados: dict) -> int:
    documentos = resultados["documents"][0]
    return sum(estimar_tokens(documento) for documento in documentos)




### 10.1 Antes da recuperação

- **Detecção de ambiguidade:** impede assumir um escopo quando a pergunta aceita
  interpretações diferentes.
- **Rewrite:** reformula a pergunta de acordo com o escopo informado.
- **Expansão:** cria variações lexicais da mesma intenção.
- **Step-back:** formula uma questão mais geral para buscar contexto conceitual.
- **HyDE:** cria um trecho hipotético parecido com a resposta esperada e o usa
  somente para recuperar documentos. O texto hipotético não é evidência e não
  deve aparecer como fonte da resposta.

Essas técnicas podem aumentar cobertura, mas também custo e ruído. Por isso as
consultas geradas devem ser registradas na atividade.


In [14]:
# ------------------------------------------------------------
# Pré-recuperação: ambiguidade e transformação de consultas
# ------------------------------------------------------------

def detectar_ambiguidade(pergunta: str) -> bool:
    p = normalizar_para_busca(pergunta)
    termos_ambiguos = [
        "frequencia minima",
        "carga horaria",
        "horas obrigatorias",
    ]
    return any(termo in p for termo in termos_ambiguos)


def reescrever_consulta(
    pergunta: str,
    escopo: str | None = None,
) -> str:
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p:
        if escopo == "disciplina":
            return (
                "frequencia minima para aprovacao "
                "em disciplina regular"
            )

        if escopo == "estagio":
            return (
                "regra de frequencia ou assiduidade "
                "no estagio obrigatorio"
            )

        if escopo == "curso":
            return (
                "regra geral de frequencia minima "
                "para aprovacao prevista no PPC do curso"
            )

        # Sem escopo não assumimos interpretação.
        return pergunta

    if "sql" in p:
        return (
            "conteudo programatico da disciplina "
            "de banco de dados SQL"
        )

    if "estagio" in p:
        return (
            "carga horaria do estagio obrigatorio "
            "supervisionado"
        )

    return pergunta


def expandir_consulta(
    pergunta: str,
    escopo: str | None = None,
) -> list[str]:
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p:
        if escopo == "disciplina":
            return [
                pergunta,
                "presenca minima para aprovacao em disciplina",
                "percentual minimo de comparecimento disciplina",
                "limite de faltas disciplina regular",
            ]

        if escopo == "estagio":
            return [
                pergunta,
                "frequencia no estagio obrigatorio",
                "assiduidade no estagio supervisionado",
                "controle de presenca no estagio obrigatorio",
            ]

        if escopo == "curso":
            return [
                pergunta,
                "regra geral de frequencia prevista no PPC",
                "criterios de aprovacao por frequencia no curso",
                "normas academicas de frequencia do curso",
            ]

        # Pergunta ainda ambígua: não expandir agressivamente.
        return [pergunta]

    if "estagio" in p:
        return [
            pergunta,
            "carga horaria estagio obrigatorio",
            "horas estagio supervisionado",
        ]

    return [pergunta]


def step_back(
    pergunta: str,
    escopo: str | None = None,
) -> str:
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p:
        if escopo == "disciplina":
            return (
                "regras academicas de aprovacao "
                "e frequencia em disciplinas"
            )

        if escopo == "estagio":
            return (
                "regras academicas de acompanhamento "
                "e assiduidade no estagio"
            )

        if escopo == "curso":
            return (
                "normas academicas gerais de aprovacao "
                "e frequencia previstas no PPC"
            )

        return "regras academicas de frequencia"

    if "estagio" in p:
        return (
            "regras dos componentes curriculares "
            "obrigatorios do curso"
        )

    return pergunta


def hyde_controlado(
    pergunta: str,
    escopo: str | None = None,
) -> str:
    """
    Simulação didática de HyDE sem chamar um LLM.

    O texto hipotético serve somente para produzir uma consulta de
    recuperação. Ele NÃO é evidência.
    """
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "estagio" in p and "frequencia" not in p:
        return (
            "O regulamento de estágio informa "
            "a carga horária do estágio obrigatório."
        )

    if "frequencia" in p:
        if escopo == "disciplina":
            return (
                "A ficha ou norma acadêmica informa "
                "a frequência mínima necessária para "
                "aprovação em uma disciplina."
            )

        if escopo == "estagio":
            return (
                "O regulamento de estágio estabelece "
                "regras de frequência e assiduidade "
                "para o estágio obrigatório."
            )

        if escopo == "curso":
            return (
                "O projeto pedagógico do curso apresenta "
                "as normas gerais de frequência e os "
                "critérios acadêmicos de aprovação."
            )

        return pergunta

    return pergunta


def gerar_consultas(
    pergunta: str,
    escopo: str | None = None,
) -> list[str]:
    validar_escopo(escopo)

    consultas = [
        pergunta,
        reescrever_consulta(pergunta, escopo=escopo),
    ]

    consultas.extend(
        expandir_consulta(pergunta, escopo=escopo)
    )

    consultas.append(
        step_back(pergunta, escopo=escopo)
    )

    consultas.append(
        hyde_controlado(pergunta, escopo=escopo)
    )

    # Remove duplicatas preservando a ordem.
    return list(dict.fromkeys(consultas))




### 10.2 Recuperação e fusão por RRF

Cada variação da consulta produz um ranking. O Reciprocal Rank Fusion soma
contribuições baseadas na posição de cada chunk. Um item bem colocado em várias
listas ganha força sem exigir que scores de consultas diferentes sejam
diretamente comparáveis.


In [15]:
# ------------------------------------------------------------
# Recuperação e fusão
# ------------------------------------------------------------

def converter_resultados(
    resultados: dict,
    consulta: str,
) -> list[ResultadoChunk]:
    itens = []

    for identificador, documento, metadata, distancia in zip(
        resultados["ids"][0],
        resultados["documents"][0],
        resultados["metadatas"][0],
        resultados["distances"][0],
    ):
        itens.append(
            ResultadoChunk(
                id=identificador,
                texto=documento,
                metadata=metadata,
                distancia=float(distancia),
                similaridade=1 - float(distancia),
                consulta_origem=consulta,
            )
        )

    return itens


def buscar_lista(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    k: int = TOP_K,
    escopo: str | None = None,
) -> list[ResultadoChunk]:
    resultados = buscar(
        collection,
        modelo,
        pergunta,
        k=k,
        escopo=escopo,
    )
    return converter_resultados(resultados, consulta=pergunta)


def fusao_rrf(
    listas: list[list[ResultadoChunk]],
    k_rrf: int = 60,
) -> list[ResultadoChunk]:
    scores = {}
    objetos = {}

    for lista in listas:
        for posicao, item in enumerate(lista, start=1):
            scores[item.id] = (
                scores.get(item.id, 0.0)
                + 1.0 / (k_rrf + posicao)
            )
            objetos[item.id] = item

    ordenados = sorted(
        scores.items(),
        key=lambda par: par[1],
        reverse=True,
    )

    return [
        objetos[identificador]
        for identificador, _ in ordenados
    ]


def multi_busca(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    escopo: str | None = None,
    k: int = 4,
) -> list[ResultadoChunk]:
    validar_escopo(escopo)
    listas = []

    consultas = gerar_consultas(
        pergunta,
        escopo=escopo,
    )

    for consulta in consultas:
        listas.append(
            buscar_lista(
                collection,
                modelo,
                consulta,
                k=k,
                escopo=escopo,
            )
        )

    return fusao_rrf(listas)




### 10.3 Filtros, deduplicação e reranking

O filtro de escopo funciona como defesa adicional. O limiar elimina candidatos
fracos. A deduplicação remove textos idênticos ou muito sobrepostos. Por fim, o
reranker combina similaridade vetorial, cobertura lexical e bônus de metadados.

Cada etapa traz um risco: um limiar alto pode apagar evidência útil; uma
deduplicação agressiva pode confundir trechos complementares; um reranker
lexical pode favorecer repetição de palavras em vez de significado.


In [16]:
# ------------------------------------------------------------
# Pós-recuperação: filtros, deduplicação e reranking
# ------------------------------------------------------------

def filtrar_por_escopo(
    candidatos: list[ResultadoChunk],
    escopo: str | None,
) -> list[ResultadoChunk]:
    """
    Filtro defensivo pós-recuperação.

    No pipeline avançado, o escopo também é usado como filtro simbólico
    durante a própria busca no ChromaDB. Mantemos esta etapa para tornar
    explícita a validação do contexto antes de enviá-lo ao LLM.
    """
    validar_escopo(escopo)

    if escopo is None:
        return candidatos

    return [
        candidato
        for candidato in candidatos
        if candidato.metadata.get("escopo") == escopo
    ]


def filtrar_por_score(
    candidatos: list[ResultadoChunk],
    minimo: float = LIMIAR_SIMILARIDADE,
) -> list[ResultadoChunk]:
    return [
        candidato
        for candidato in candidatos
        if candidato.similaridade >= minimo
    ]


def hash_texto(texto: str) -> str:
    normalizado = normalizar_trecho_contexto(
        normalizar_para_busca(texto)
    )
    return hashlib.md5(
        normalizado.encode("utf-8")
    ).hexdigest()[:12]


def deduplicar_textual(
    candidatos: list[ResultadoChunk],
) -> list[ResultadoChunk]:
    vistos = set()
    saida = []

    for candidato in candidatos:
        h = hash_texto(candidato.texto)

        if h not in vistos:
            vistos.add(h)
            saida.append(candidato)

    return saida


def termos_conteudo(texto: str) -> set[str]:
    stopwords = {
        "qual", "quais", "como", "para", "possui", "sobre",
        "uma", "um", "de", "do", "da", "das", "dos", "e",
        "o", "a", "as", "os", "em", "no", "na", "nos", "nas",
        "que", "se", "ao", "aos", "ou", "por", "com",
    }

    termos = re.findall(
        r"\w+",
        normalizar_para_busca(texto),
    )

    return {
        termo
        for termo in termos
        if termo not in stopwords and len(termo) > 2
    }


def deduplicar_semantico_lexical(
    candidatos: list[ResultadoChunk],
    limiar_jaccard: float = 0.82,
) -> list[ResultadoChunk]:
    """
    Deduplicação aproximada para a demonstração.

    Usa sobreposição lexical. Em produção, pode ser substituída por
    similaridade entre embeddings dos próprios chunks.
    """
    saida = []
    assinaturas = []

    for candidato in candidatos:
        termos = termos_conteudo(candidato.texto)
        duplicado = False

        for assinatura in assinaturas:
            inter = len(termos & assinatura)
            union = len(termos | assinatura) or 1
            jaccard = inter / union

            if jaccard >= limiar_jaccard:
                duplicado = True
                break

        if not duplicado:
            saida.append(candidato)
            assinaturas.append(termos)

    return saida


def deduplicar(
    candidatos: list[ResultadoChunk],
) -> list[ResultadoChunk]:
    candidatos = deduplicar_textual(candidatos)
    candidatos = deduplicar_semantico_lexical(candidatos)
    return candidatos


def palavras_relevantes(pergunta: str) -> set[str]:
    return termos_conteudo(pergunta)


def rerank_lexical(
    pergunta: str,
    candidatos: list[ResultadoChunk],
) -> list[ResultadoChunk]:
    """
    Reranker didático sem modelo adicional.

    Combina similaridade vetorial, cobertura lexical e pequenos bônus
    por presença de fonte/metadado controlado.
    """
    termos = palavras_relevantes(pergunta)

    def score(candidato: ResultadoChunk) -> float:
        texto = normalizar_para_busca(candidato.texto)
        cobertura = sum(
            1
            for termo in termos
            if termo in texto
        )

        bonus_fonte = (
            0.05
            if candidato.metadata.get("arquivo")
            else 0.0
        )

        bonus_controlado = (
            0.05
            if candidato.metadata.get("fonte_controlada")
            else 0.0
        )

        return (
            candidato.similaridade
            + 0.12 * cobertura
            + bonus_fonte
            + bonus_controlado
        )

    return sorted(
        candidatos,
        key=score,
        reverse=True,
    )




### 10.4 Compressão e orçamento de contexto

O contexto disputa a janela do modelo com instruções, histórico, pergunta,
metadados e espaço para a resposta. A compressão extrativa mantém sentenças
relacionadas à pergunta até o orçamento definido.

Ao reduzir texto, preserve cinco elementos: valor, escopo, condição, exceção e
fonte. Um contexto menor pode ter custo melhor e confiabilidade pior se perder
qualquer um deles.


In [17]:
# ------------------------------------------------------------
# Compressão de contexto
# ------------------------------------------------------------

def sentencas_relevantes(
    pergunta: str,
    texto: str,
    limite: int = 3,
) -> list[str]:
    termos = palavras_relevantes(pergunta)
    texto = normalizar_trecho_contexto(texto)

    sentencas = re.split(
        r"(?<=[.!?])\s+",
        texto.strip(),
    )

    selecionadas = [
        sentenca
        for sentenca in sentencas
        if any(
            termo in normalizar_para_busca(sentenca)
            for termo in termos
        )
    ]

    return selecionadas[:limite] or sentencas[:1]


def montar_prompt_compressao_abstrativa(
    pergunta: str,
    trecho: str,
) -> str:
    """
    Prompt de exemplo para uma compressão abstrativa com LLM.

    A demonstração executa compressão extrativa para não depender de API.
    """
    return f"""
Resuma o TRECHO apenas para responder à PERGUNTA.
Preserve números, condições, exceções e fonte.
Não acrescente informação externa.
Se o trecho não ajudar, responda: IRRELEVANTE.

PERGUNTA:
{pergunta}

TRECHO:
{trecho}
""".strip()


def comprimir_contexto(
    pergunta: str,
    candidatos: list[ResultadoChunk],
    limite_tokens: int = LIMITE_TOKENS_CONTEXTO,
) -> tuple[list[str], int]:
    contexto = []
    usados = 0

    for candidato in candidatos:
        trecho = " ".join(
            sentencas_relevantes(
                pergunta,
                candidato.texto,
            )
        )

        trecho = normalizar_trecho_contexto(trecho)

        fonte = (
            f"{candidato.metadata.get('arquivo')} "
            f"p.{candidato.metadata.get('pagina')} "
            f"escopo={candidato.metadata.get('escopo')}"
        )

        item = f"Fonte: {fonte}. Trecho: {trecho}"
        custo = estimar_tokens(item)

        if usados + custo <= limite_tokens:
            contexto.append(item)
            usados += custo

    return contexto, usados




### 10.5 Suficiência e decisão

O pipeline verifica se o contexto cobre os termos centrais. Perguntas
quantitativas também exigem número ou percentual explícito. Dependendo do caso,
a ação segura pode ser esclarecer, buscar novamente, abster-se ou responder.

Na atividade, não avalie suficiência apenas pelo score: leia o trecho e confirme
se ele sustenta a resposta sem conhecimento externo.


In [18]:
# ------------------------------------------------------------
# Contexto suficiente e decisão de resposta
# ------------------------------------------------------------

def contexto_suficiente(
    pergunta: str,
    contexto: list[str],
) -> tuple[bool, str]:
    if not contexto:
        return False, "nenhum contexto foi selecionado"

    texto_contexto = " ".join(contexto)
    texto_normalizado = normalizar_para_busca(texto_contexto)
    termos = palavras_relevantes(pergunta)

    cobertura = sum(
        1
        for termo in termos
        if termo in texto_normalizado
    )

    if cobertura == 0:
        return (
            False,
            "o contexto não cobre os termos centrais da pergunta",
        )

    p = normalizar_para_busca(pergunta)

    # Perguntas quantitativas precisam de um valor explícito.
    if any(
        termo in p
        for termo in [
            "quantas",
            "quanto",
            "carga horaria",
            "horas",
            "frequencia",
        ]
    ):
        possui_numero = bool(
            re.search(r"\b\d+(?:[.,]\d+)?\b", texto_contexto)
        )
        possui_percentual = "%" in texto_contexto

        if not (possui_numero or possui_percentual):
            return (
                False,
                "a pergunta exige um valor, mas o contexto não contém número ou percentual",
            )

    return True, "há evidência mínima para responder"


def montar_prompt(
    pergunta: str,
    contexto: list[str],
) -> str:
    evidencias = "\n".join(
        f"[{indice}] {conteudo}"
        for indice, conteudo in enumerate(
            contexto,
            start=1,
        )
    )

    return f"""
Responda somente com base no CONTEXTO.
Se o contexto não for suficiente, diga que não há evidência suficiente.
Não complete lacunas com conhecimento externo.
Cite a fonte usada.

PERGUNTA:
{pergunta}

CONTEXTO:
{evidencias}
""".strip()


def sugerir_fonte_estruturada(pergunta: str) -> str:
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p or "carga horaria" in p or "estagio" in p:
        return "metadados/regras acadêmicas estruturadas"

    if "disciplina" in p or "sql" in p:
        return "consulta SQL no catálogo de disciplinas"

    return "documentos recuperados + metadados controlados"




### 10.6 Consultas geradas

Esta função torna visíveis as transformações feitas antes da recuperação. Copie
essas consultas para o relatório: elas ajudam a explicar aumento de cobertura,
ruído ou latência.


In [19]:
# ------------------------------------------------------------
# Exibição das transformações
# ------------------------------------------------------------

def imprimir_consultas_geradas(
    pergunta: str,
    escopo: str | None,
) -> None:
    print("\nConsultas que serão executadas:")

    for indice, consulta in enumerate(
        gerar_consultas(pergunta, escopo),
        start=1,
    ):
        print(f"  {indice}. {consulta}")




### 10.7 Pipeline avançado completo

`responder_com_contexto` encadeia as etapas anteriores e mede seus tempos. O
resultado final inclui contagens intermediárias, tokens, decisão e o prompt que
seria enviado ao LLM. Ele não chama uma API de LLM; o foco da aula é avaliar a
qualidade do contexto construído.


In [20]:
# ------------------------------------------------------------
# Pipeline final com medição de custo por etapa
# ------------------------------------------------------------

def responder_com_contexto(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    escopo: str | None = None,
) -> str:
    validar_escopo(escopo)
    pergunta = pergunta.strip()

    if not pergunta:
        raise ValueError("A pergunta não pode estar vazia.")

    # --------------------------------------------------------
    # 1. Decisão ANTES da recuperação
    # --------------------------------------------------------
    if detectar_ambiguidade(pergunta) and escopo is None:
        return (
            "Acao: esclarecer\n"
            "Motivo: pergunta ambigua sem escopo definido\n\n"
            "A pergunta pode se referir a diferentes regras.\n"
            "Voce se refere a:\n"
            "1. disciplina;\n"
            "2. estagio;\n"
            "3. regra geral prevista no PPC do curso?"
        )

    tempo_total = perf_counter()
    tempos = {}

    # --------------------------------------------------------
    # 2. Transformação + múltiplas buscas + RRF
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = multi_busca(
        collection,
        modelo,
        pergunta,
        escopo=escopo,
        k=4,
    )

    tempos["multi_busca_rrf_ms"] = (
        perf_counter() - t
    ) * 1000

    total_pos_fusao = len(candidatos)

    # --------------------------------------------------------
    # 3. Filtros
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = filtrar_por_escopo(
        candidatos,
        escopo,
    )

    candidatos = filtrar_por_score(
        candidatos,
        minimo=LIMIAR_SIMILARIDADE,
    )

    tempos["filtros_ms"] = (
        perf_counter() - t
    ) * 1000

    total_pos_filtros = len(candidatos)

    # --------------------------------------------------------
    # 4. Deduplicação
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = deduplicar(candidatos)

    tempos["deduplicacao_ms"] = (
        perf_counter() - t
    ) * 1000

    total_pos_dedup = len(candidatos)

    # --------------------------------------------------------
    # 5. Reranking
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = rerank_lexical(
        pergunta,
        candidatos,
    )

    tempos["reranking_ms"] = (
        perf_counter() - t
    ) * 1000

    # --------------------------------------------------------
    # 6. Compressão
    # --------------------------------------------------------
    t = perf_counter()

    contexto, tokens = comprimir_contexto(
        pergunta,
        candidatos[:5],
        limite_tokens=LIMITE_TOKENS_CONTEXTO,
    )

    tempos["compressao_ms"] = (
        perf_counter() - t
    ) * 1000

    # --------------------------------------------------------
    # 7. Contexto suficiente?
    # --------------------------------------------------------
    suficiente, motivo = contexto_suficiente(
        pergunta,
        contexto,
    )

    latencia_ms = (
        perf_counter() - tempo_total
    ) * 1000

    linhas_tempo = "\n".join(
        f"- {nome}: {valor:.2f} ms"
        for nome, valor in tempos.items()
    )

    cabecalho = (
        f"Escopo: {escopo}\n"
        f"Candidatos apos fusao: {total_pos_fusao}\n"
        f"Candidatos apos filtros: {total_pos_filtros}\n"
        f"Candidatos apos deduplicacao: {total_pos_dedup}\n"
        f"Tokens aproximados do contexto: {tokens}\n"
        f"Latencia total aproximada: {latencia_ms:.2f} ms\n"
        f"Fonte estruturada sugerida: "
        f"{sugerir_fonte_estruturada(pergunta)}\n"
        f"\nTempos por etapa:\n{linhas_tempo}\n"
    )

    if not suficiente:
        return (
            cabecalho
            + "\nAcao: abster\n"
            + f"Motivo: {motivo}\n"
            + "Nao ha evidencia suficiente no contexto recuperado."
        )

    prompt = montar_prompt(
        pergunta,
        contexto,
    )

    return (
        cabecalho
        + "\nAcao: responder\n"
        + f"Motivo: {motivo}\n\n"
        + prompt
    )




## 11. Programa interativo original

A função `main()` carrega PDFs, cria chunks, instancia o modelo, prepara a
coleção e abre o terminal interativo. A definição permanece abaixo, mas sua
chamada foi deixada no final do notebook. Para a atividade, use primeiro o
roteiro guiado, que mantém as variáveis acessíveis entre as células.


In [21]:
# ============================================================
# PROGRAMA PRINCIPAL
# ============================================================

def main() -> None:
    try:
        print("1. Carregando PDFs...")
        paginas = carregar_pdfs(PASTA_DOCUMENTOS)
        print(f"Páginas com texto: {len(paginas)}")

        print("\n2. Gerando chunks...")
        chunks = preparar_chunks(paginas)
        print(f"Total de chunks: {len(chunks)}")

        print("\n3. Carregando o modelo de embeddings...")
        modelo = SentenceTransformer(MODELO_EMBEDDING)

        print(f"Modelo: {MODELO_EMBEDDING}")
        print(
            "Dimensão dos embeddings: "
            f"{modelo.get_sentence_embedding_dimension()}"
        )

        print("\n4. Preparando a coleção...")
        collection = criar_colecao(
            recriar=RECRIAR_COLECAO
        )

        if RECRIAR_COLECAO or collection.count() == 0:
            print("\n5. Indexando os chunks...")
            indexar_chunks(
                collection,
                modelo,
                chunks,
            )
        else:
            print(
                "\nColeção persistente reutilizada. "
                f"Registros: {collection.count()}"
            )

        print("\n6. Busca interativa")
        print("Comandos disponíveis:")
        print("  modo baseline")
        print("  modo avancado")
        print("  escopo disciplina")
        print("  escopo estagio")
        print("  escopo curso")
        print("  escopo nenhum")
        print("  mostrar consultas")
        print("  sair")

        modo_pipeline = MODO_PIPELINE
        escopo_atual = ESCOPO_PADRAO
        mostrar_consultas = True

        while True:
            pergunta = input(
                "\nDigite sua opção ou pergunta ou 'sair': "
            ).strip()

            if pergunta.lower() == "sair":
                print("Programa encerrado.")
                break

            comando = normalizar_para_busca(pergunta)

            if comando == "modo baseline":
                modo_pipeline = "baseline"
                print("Modo alterado para baseline.")
                continue

            if comando == "modo avancado":
                modo_pipeline = "avancado"
                print("Modo alterado para avançado.")
                continue

            if comando == "mostrar consultas":
                mostrar_consultas = not mostrar_consultas
                print(
                    "Exibição das consultas geradas: "
                    f"{mostrar_consultas}"
                )
                continue

            if comando.startswith("escopo "):
                valor = comando.replace(
                    "escopo ",
                    "",
                    1,
                ).strip()

                novo_escopo = (
                    None
                    if valor == "nenhum"
                    else valor
                )

                validar_escopo(novo_escopo)
                escopo_atual = novo_escopo
                print(f"Escopo atual: {escopo_atual}")
                continue

            if not pergunta:
                print("Digite uma pergunta válida.")
                continue

            # ------------------------------------------------
            # PIPELINE AVANÇADO
            # ------------------------------------------------
            if modo_pipeline == "avancado":
                if (
                    mostrar_consultas
                    and not (
                        detectar_ambiguidade(pergunta)
                        and escopo_atual is None
                    )
                ):
                    imprimir_consultas_geradas(
                        pergunta,
                        escopo_atual,
                    )

                print(
                    responder_com_contexto(
                        collection,
                        modelo,
                        pergunta,
                        escopo=escopo_atual,
                    )
                )
                continue

            # ------------------------------------------------
            # BASELINE DA AULA ANTERIOR
            # ------------------------------------------------
            if MODO_BUSCA == "mmr":
                resultados = buscar_mmr(
                    collection,
                    modelo,
                    pergunta,
                    k=TOP_K,
                    fetch_k=FETCH_K_MMR,
                    lambda_mult=LAMBDA_MMR,
                    escopo=None,
                )
            else:
                resultados = buscar(
                    collection,
                    modelo,
                    pergunta,
                    k=TOP_K,
                    escopo=None,
                )

            imprimir_resultados(resultados)

            print(
                "Tokens aproximados dos chunks recuperados: "
                f"{estimar_tokens_resultados(resultados)}"
            )

    except KeyboardInterrupt:
        print("\nPrograma interrompido.")

    except (
        FileNotFoundError,
        ValueError,
        RuntimeError,
    ) as erro:
        print(f"\nErro: {erro}")

    except Exception as erro:
        print(
            "\nErro inesperado: "
            f"{type(erro).__name__}: {erro}"
        )


# Roteiro guiado da atividade

Execute esta parte depois de executar todas as definições anteriores. Use a
mesma pergunta nas três configurações. Isso evita comparar resultados causados
por perguntas diferentes.


## A. Defina pergunta e escopo

Escolha uma pergunta diferente das já demonstradas em aula e que possa ser
respondida pelos documentos. Preencha também a justificativa em texto. O
escopo deve ser exatamente `curso`, `disciplina` ou `estagio`.


In [22]:
PERGUNTA_ATIVIDADE = (
    "Qual é a carga horária do estágio obrigatório e que exigência sobre a "
    "conclusão das disciplinas básicas deve ser cumprida para realizá-lo?"
)  # Preencha sem copiar uma pergunta já executada em aula.
ESCOPO_ATIVIDADE = "estagio"  # Troque por: "curso", "disciplina" ou "estagio".
JUSTIFICATIVA_ESCOPO = (
    "A pergunta trata das regras e da carga horária do estágio obrigatório. "
    "Essas informações pertencem ao regulamento de estágio, portanto o "
    "escopo esperado é estagio."
)  # Explique por que esse documento/escopo é o adequado.

if not PERGUNTA_ATIVIDADE.strip():
    raise ValueError("Preencha PERGUNTA_ATIVIDADE antes de continuar.")

validar_escopo(ESCOPO_ATIVIDADE)

if ESCOPO_ATIVIDADE is None:
    raise ValueError("Defina o escopo esperado para a atividade.")

print(f"Pergunta: {PERGUNTA_ATIVIDADE}")
print(f"Escopo esperado: {ESCOPO_ATIVIDADE}")
print(f"Justificativa: {JUSTIFICATIVA_ESCOPO}")


Pergunta: Qual é a carga horária do estágio obrigatório e que exigência sobre a conclusão das disciplinas básicas deve ser cumprida para realizá-lo?
Escopo esperado: estagio
Justificativa: A pergunta trata das regras e da carga horária do estágio obrigatório. Essas informações pertencem ao regulamento de estágio, portanto o escopo esperado é estagio.


## B. Prepare o índice uma única vez

Esta célula usa as mesmas funções e parâmetros do `main()`. Depois da primeira
indexação bem-sucedida, você pode definir `RECRIAR_COLECAO = False` e reutilizar
o banco. O download inicial do modelo exige internet.


In [23]:
paginas_atividade = carregar_pdfs(PASTA_DOCUMENTOS)
chunks_atividade = preparar_chunks(paginas_atividade)

modelo_atividade = SentenceTransformer(MODELO_EMBEDDING)
colecao_atividade = criar_colecao(recriar=RECRIAR_COLECAO)

if RECRIAR_COLECAO or colecao_atividade.count() == 0:
    indexar_chunks(
        colecao_atividade,
        modelo_atividade,
        chunks_atividade,
    )

print(f"Páginas: {len(paginas_atividade)}")
print(f"Chunks: {len(chunks_atividade)}")
print(f"Registros na coleção: {colecao_atividade.count()}")


Lendo: 2024-regulamento-estagio.pdf
Lendo: PPC-do-Curso-de-Ciencia-da-Computação.pdf


Lendo: ci1218.pdf
  Aviso: página 3 sem texto extraível.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12067.58it/s]

Páginas: 37
Chunks: 193
Registros na coleção: 193


## C. Execute o baseline

O enunciado pede busca por similaridade. Por isso esta célula chama `buscar`,
não `buscar_mmr`. Guarde os três resultados e julgue manualmente se o top-1
contém evidência suficiente.


In [24]:
resultado_baseline_atividade = buscar(
    colecao_atividade,
    modelo_atividade,
    PERGUNTA_ATIVIDADE,
    k=TOP_K,
    escopo=None,
)

imprimir_resultados(resultado_baseline_atividade)

print(
    "Tokens aproximados: ",
    estimar_tokens_resultados(resultado_baseline_atividade),
)
print(
    "Latência da recuperação: "
    f"{resultado_baseline_atividade['latencia_ms']:.2f} ms"
)
print(
    "Escopos recuperados: ",
    [
        metadata.get("escopo")
        for metadata in resultado_baseline_atividade["metadatas"][0]
    ],
)



################################################################################
Pergunta: Qual é a carga horária do estágio obrigatório e que exigência sobre a conclusão das disciplinas básicas deve ser cumprida para realizá-lo?
Método: similaridade
Escopo: None
Latência da busca: 2.47 ms
################################################################################

Rank: 1
ID: PPC-do-Curso-de-Ciencia-da-Computação_p024_c002
Arquivo: PPC-do-Curso-de-Ciencia-da-Computação.pdf
Página: 24
Chunk: 2
Escopo: curso
Tipo: ppc
Distância de cosseno: 0.1730
Similaridade aproximada: 0.8270
--------------------------------------------------------------------------------
r estágio obrigatório
com carga horária de duzentas e vinte horas (220 horas), mediante matrícula na disciplina de Estágio
Obrigatório, para fins de integralização curricular.
Art. 15º A disciplina de Estágio Obrigatório deverá ser realizada após a conclusão das disciplinas básicas
da grade curricular, conforme consta no Pr

## D. Observe as etapas do pipeline avançado

A função auxiliar abaixo não substitui nem modifica o pipeline. Ela apenas chama
as funções existentes na mesma ordem e devolve cópias das listas intermediárias,
permitindo registrar candidatos, filtros, deduplicação e reranking.


In [25]:
def observar_etapas_pipeline(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    escopo: str | None,
) -> dict[str, list[ResultadoChunk]]:
    """Expõe resultados intermediários sem alterar as funções do pipeline."""
    apos_fusao = multi_busca(
        collection,
        modelo,
        pergunta,
        escopo=escopo,
        k=4,
    )
    apos_filtro_escopo = filtrar_por_escopo(apos_fusao, escopo)
    apos_filtros = filtrar_por_score(
        apos_filtro_escopo,
        minimo=LIMIAR_SIMILARIDADE,
    )
    apos_deduplicacao = deduplicar(apos_filtros)
    apos_reranking = rerank_lexical(pergunta, apos_deduplicacao)

    return {
        "fusão": apos_fusao,
        "filtros": apos_filtros,
        "deduplicação": apos_deduplicacao,
        "reranking": apos_reranking,
    }


def imprimir_etapas_pipeline(etapas: dict[str, list[ResultadoChunk]]) -> None:
    for nome, candidatos in etapas.items():
        print("\n" + "=" * 80)
        print(f"Etapa: {nome} | candidatos: {len(candidatos)}")
        for posicao, candidato in enumerate(candidatos, start=1):
            print(
                f"{posicao}. {candidato.id} | "
                f"similaridade={candidato.similaridade:.4f} | "
                f"escopo={candidato.metadata.get('escopo')} | "
                f"arquivo={candidato.metadata.get('arquivo')}"
            )


In [26]:
etapas_atividade = observar_etapas_pipeline(
    colecao_atividade,
    modelo_atividade,
    PERGUNTA_ATIVIDADE,
    ESCOPO_ATIVIDADE,
)

imprimir_etapas_pipeline(etapas_atividade)



Etapa: fusão | candidatos: 11
1. 2024-regulamento-estagio_p003_c001 | similaridade=0.6355 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
2. 2024-regulamento-estagio_p004_c001 | similaridade=0.6177 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
3. 2024-regulamento-estagio_p004_c005 | similaridade=0.5788 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
4. 2024-regulamento-estagio_p003_c003 | similaridade=0.8068 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
5. 2024-regulamento-estagio_p005_c001 | similaridade=0.5201 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
6. 2024-regulamento-estagio_p002_c004 | similaridade=0.5055 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
7. 2024-regulamento-estagio_p004_c003 | similaridade=0.6201 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
8. 2024-regulamento-estagio_p003_c004 | similaridade=0.7209 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
9. 2024-regulamento-estagio_p002_c005 | s

## E. Execute o pipeline avançado

Registre as consultas, as contagens, os tokens, a latência, a decisão e o
contexto final. Não conclua apenas pelo número de candidatos: leia as fontes e
verifique se o contexto preserva valores, condições, exceções e escopo.


In [27]:
imprimir_consultas_geradas(
    PERGUNTA_ATIVIDADE,
    ESCOPO_ATIVIDADE,
)

saida_avancada_atividade = responder_com_contexto(
    colecao_atividade,
    modelo_atividade,
    PERGUNTA_ATIVIDADE,
    escopo=ESCOPO_ATIVIDADE,
)

print(saida_avancada_atividade)



Consultas que serão executadas:
  1. Qual é a carga horária do estágio obrigatório e que exigência sobre a conclusão das disciplinas básicas deve ser cumprida para realizá-lo?
  2. carga horaria do estagio obrigatorio supervisionado
  3. carga horaria estagio obrigatorio
  4. horas estagio supervisionado
  5. regras dos componentes curriculares obrigatorios do curso
  6. O regulamento de estágio informa a carga horária do estágio obrigatório.
Escopo: estagio
Candidatos apos fusao: 11
Candidatos apos filtros: 11
Candidatos apos deduplicacao: 11
Tokens aproximados do contexto: 598
Latencia total aproximada: 43.83 ms
Fonte estruturada sugerida: metadados/regras acadêmicas estruturadas

Tempos por etapa:
- multi_busca_rrf_ms: 40.10 ms
- filtros_ms: 0.00 ms
- deduplicacao_ms: 1.74 ms
- reranking_ms: 0.63 ms
- compressao_ms: 1.09 ms

Acao: responder
Motivo: há evidência mínima para responder

Responda somente com base no CONTEXTO.
Se o contexto não for suficiente, diga que não há evidência 

### Registro das transformações de consulta

O enunciado pede observar *rewrite*, expansão, *step-back* e HyDE. A célula
seguinte registra o resultado de cada técnica **antes** da deduplicação. Em
perguntas para as quais uma técnica não possui regra específica, ela pode
devolver a própria pergunta original; isso também é um resultado válido e deve
ser relatado.


In [28]:
TRANSFORMACOES_CONSULTA = {
    "Consulta original": [PERGUNTA_ATIVIDADE],
    "Rewrite": [
        reescrever_consulta(PERGUNTA_ATIVIDADE, escopo=ESCOPO_ATIVIDADE)
    ],
    "Expansão": expandir_consulta(
        PERGUNTA_ATIVIDADE,
        escopo=ESCOPO_ATIVIDADE,
    ),
    "Step-back": [
        step_back(PERGUNTA_ATIVIDADE, escopo=ESCOPO_ATIVIDADE)
    ],
    "HyDE controlado": [
        hyde_controlado(PERGUNTA_ATIVIDADE, escopo=ESCOPO_ATIVIDADE)
    ],
}

CONSULTAS_UNICAS = gerar_consultas(
    PERGUNTA_ATIVIDADE,
    escopo=ESCOPO_ATIVIDADE,
)

for tecnica, consultas in TRANSFORMACOES_CONSULTA.items():
    print(f"{tecnica}: {consultas}")

print("\nConsultas únicas após deduplicação:")
for indice, consulta in enumerate(CONSULTAS_UNICAS, start=1):
    print(f"{indice}. {consulta}")


Consulta original: ['Qual é a carga horária do estágio obrigatório e que exigência sobre a conclusão das disciplinas básicas deve ser cumprida para realizá-lo?']
Rewrite: ['carga horaria do estagio obrigatorio supervisionado']
Expansão: ['Qual é a carga horária do estágio obrigatório e que exigência sobre a conclusão das disciplinas básicas deve ser cumprida para realizá-lo?', 'carga horaria estagio obrigatorio', 'horas estagio supervisionado']
Step-back: ['regras dos componentes curriculares obrigatorios do curso']
HyDE controlado: ['O regulamento de estágio informa a carga horária do estágio obrigatório.']

Consultas únicas após deduplicação:
1. Qual é a carga horária do estágio obrigatório e que exigência sobre a conclusão das disciplinas básicas deve ser cumprida para realizá-lo?
2. carga horaria do estagio obrigatorio supervisionado
3. carga horaria estagio obrigatorio
4. horas estagio supervisionado
5. regras dos componentes curriculares obrigatorios do curso
6. O regulamento de 

## F. Faça uma alteração controlada

Escolha apenas uma mudança. As opções mais diretas, sem redefinir funções, são:

- `LIMIAR_SIMILARIDADE = 0.40`;
- `LIMITE_TOKENS_CONTEXTO = 300`;
- executar com e sem `ESCOPO_ATIVIDADE`.

Também é possível testar `LAMBDA_MMR`, mas isso deve ser feito chamando
`buscar_mmr`; essa variável não participa do pipeline avançado atual. Remover
reranking ou deduplicação exige alterar a composição do pipeline e deve ser
documentado explicitamente.

Na célula seguinte, descreva e aplique uma única alteração. Os valores originais
ficam guardados para restauração.


In [29]:
VALORES_ORIGINAIS = {
    "LIMIAR_SIMILARIDADE": LIMIAR_SIMILARIDADE,
    "LIMITE_TOKENS_CONTEXTO": LIMITE_TOKENS_CONTEXTO,
    "LAMBDA_MMR": LAMBDA_MMR,
}

ALTERACAO_DESCRICAO = (
    "Redução do orçamento máximo de contexto de 700 para 200 tokens, "
    "mantendo todas as demais configurações."
)  # Descreva a única alteração escolhida.
ESCOPO_EXPERIMENTO = ESCOPO_ATIVIDADE

# Escolha somente UMA opção e remova o comentário da linha correspondente:
# LIMIAR_SIMILARIDADE = 0.40
LIMITE_TOKENS_CONTEXTO = 200
# ESCOPO_EXPERIMENTO = None

if not ALTERACAO_DESCRICAO.strip():
    raise ValueError(
        "Descreva e aplique uma alteração antes de executar o experimento."
    )

print(f"Alteração escolhida: {ALTERACAO_DESCRICAO}")


Alteração escolhida: Redução do orçamento máximo de contexto de 700 para 200 tokens, mantendo todas as demais configurações.


In [30]:
etapas_alteradas = observar_etapas_pipeline(
    colecao_atividade,
    modelo_atividade,
    PERGUNTA_ATIVIDADE,
    ESCOPO_EXPERIMENTO,
)
imprimir_etapas_pipeline(etapas_alteradas)

saida_alterada_atividade = responder_com_contexto(
    colecao_atividade,
    modelo_atividade,
    PERGUNTA_ATIVIDADE,
    escopo=ESCOPO_EXPERIMENTO,
)
print(saida_alterada_atividade)



Etapa: fusão | candidatos: 11
1. 2024-regulamento-estagio_p003_c001 | similaridade=0.6355 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
2. 2024-regulamento-estagio_p004_c001 | similaridade=0.6177 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
3. 2024-regulamento-estagio_p004_c005 | similaridade=0.5788 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
4. 2024-regulamento-estagio_p003_c003 | similaridade=0.8068 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
5. 2024-regulamento-estagio_p005_c001 | similaridade=0.5201 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
6. 2024-regulamento-estagio_p002_c004 | similaridade=0.5055 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
7. 2024-regulamento-estagio_p004_c003 | similaridade=0.6201 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
8. 2024-regulamento-estagio_p003_c004 | similaridade=0.7209 | escopo=estagio | arquivo=2024-regulamento-estagio.pdf
9. 2024-regulamento-estagio_p002_c005 | s

### Restaurar parâmetros

Execute após coletar os resultados alterados. Isso evita que uma nova execução
do baseline ou avançado use configurações diferentes sem você perceber.


In [31]:
LIMIAR_SIMILARIDADE = VALORES_ORIGINAIS["LIMIAR_SIMILARIDADE"]
LIMITE_TOKENS_CONTEXTO = VALORES_ORIGINAIS["LIMITE_TOKENS_CONTEXTO"]
LAMBDA_MMR = VALORES_ORIGINAIS["LAMBDA_MMR"]

print("Parâmetros originais restaurados.")


Parâmetros originais restaurados.


## G. Tabela comparativa — preencher com os resultados observados

| Critério | Baseline | Avançado | Alterado |
|---|---:|---:|---:|
| Top-1 correto? | _preencher_ | _preencher_ | _preencher_ |
| Escopo correto? | _preencher_ | _preencher_ | _preencher_ |
| Quantidade de candidatos | _preencher_ | _preencher_ | _preencher_ |
| Tokens aproximados | _preencher_ | _preencher_ | _preencher_ |
| Latência | _preencher_ | _preencher_ | _preencher_ |
| Há ruído? | _preencher_ | _preencher_ | _preencher_ |
| Contexto suficiente? | _preencher_ | _preencher_ | _preencher_ |

Use números quando existirem, mas justifique avaliações qualitativas como
“correto”, “ruído” e “suficiente” citando arquivo, página e trecho.


## H. Preencha a análise qualitativa

As métricas numéricas serão coletadas automaticamente. Complete os campos
abaixo com sua leitura das fontes e dos trechos. Respostas como “sim” ou “não”
devem ser justificadas nas observações e na análise final.

O gerador interrompe a exportação se algum campo obrigatório ficar vazio. Isso
evita produzir acidentalmente um relatório incompleto.


In [32]:
IDENTIFICACAO = {
    "nome": "Marco Antônio Mazza Canedo dos Santos",
    "turma": "MBA GenAi",
}

AVALIACAO_MANUAL = {
    "baseline": {
        "top1_correto": "Sim",
        "escopo_correto": "Não",
        "ha_ruido": "Sim; trechos adicionais sobre TCC",
        "redundancia": (
            "Sim; a informação de 220 horas aparece em mais de um resultado"
        ),
        "contexto_suficiente": "Sim",
        "observacoes": (
            "O primeiro resultado, PPC-do-Curso-de-Ciencia-da-Computação.pdf "
            "p. 24, informa as 220 horas e a conclusão das disciplinas básicas. "
            "O metadado curso difere do escopo esperado estagio, mas o PPC "
            "contém evidência pertinente. O ruído está nos trechos adicionais "
            "sobre TCC, não na classificação do documento em si."
        ),
    },
    "avancado": {
        "top1_correto": "Sim",
        "escopo_correto": "Sim",
        "ha_ruido": "Sim",
        "redundancia": (
            "Sim; as 220 horas aparecem no Art. 14 e novamente no trecho "
            "sobre validação de PET"
        ),
        "contexto_suficiente": "Sim",
        "observacoes": (
            "A busca com escopo estagio recuperou o regulamento. O trecho "
            "dos Art. 14 e 15 já era o primeiro após a fusão e o reranking "
            "manteve essa posição. O contexto, "
            "porém, inclui trechos sobre estágio não obrigatório, validação de "
            "PET e procedimentos gerais que não respondem diretamente à pergunta."
        ),
    },
    "alterado": {
        "top1_correto": "Sim",
        "escopo_correto": "Sim",
        "ha_ruido": "Não",
        "redundancia": "Não",
        "contexto_suficiente": "Sim",
        "observacoes": (
            "Com orçamento de 200 tokens, o contexto caiu de 598 para 132 tokens. "
            "Apenas o trecho do regulamento de estágio p. 3 permaneceu e ele "
            "contém tanto a carga horária de 220 horas quanto a exigência de "
            "conclusão das disciplinas básicas."
        ),
    },
}

RESPOSTAS_FINAIS = {
    "pipeline_melhor": (
        "Parcialmente. O baseline já recuperou uma resposta correta no top-1 e "
        "teve menor latência. O avançado melhorou a adequação do escopo ao usar "
        "o regulamento de estágio, mas introduziu trechos sobre estágio não "
        "obrigatório, PET e procedimentos gerais. A melhoria inequívoca ocorreu "
        "somente após limitar o contexto a 200 tokens."
    ),
    "etapa_influente": (
        "A redução do orçamento foi a alteração controlada que demonstrou "
        "melhoria na seleção final: preservou a resposta e retirou trechos "
        "desnecessários. O escopo restringiu a busca ao regulamento, mas o "
        "reranking apenas manteve o trecho dos Art. 14 e 15 no top-1, posição "
        "que ele já ocupava após a fusão. Não foram isolados experimentalmente "
        "os efeitos de todas as demais etapas."
    ),
    "evidencia_ou_ruido": (
        "As consultas adicionais recuperaram ruído, como estágio não obrigatório "
        "e validação de PET. A redução para 200 tokens não eliminou evidência útil "
        "nesta pergunta, pois o primeiro trecho contém os dois fatos necessários; "
        "em perguntas mais amplas, o mesmo limite poderia omitir informação relevante."
    ),
    "configuracao_real": (
        "Usaria o pipeline avançado com escopo estagio, expansão de consulta, "
        "fusão RRF, reranking e orçamento de 200 tokens para esta pergunta. Em "
        "produção, o orçamento deve ser calibrado por tipo de pergunta, pois 200 "
        "tokens pode ser insuficiente para respostas que exigem várias regras."
    ),
    "contexto_para_llm": (
        "Enviaria somente 2024-regulamento-estagio.pdf p. 3, Art. 14 e Art. 15. "
        "O trecho afirma diretamente que o estágio obrigatório tem 220 horas e "
        "deve ser realizado após a conclusão das disciplinas básicas da grade."
    ),
}

CONCLUSAO = (
    "A configuração avançada com escopo estagio, reranking e orçamento de 200 "
    "tokens produziu o contexto mais conciso e suficiente entre as três "
    "configurações comparadas, para a formulação final desta pergunta. Ela preservou a regra "
    "correta do regulamento de estágio e reduziu o contexto de 598 para 132 "
    "tokens. Antes da compressão, o avançado melhorou o escopo, mas não foi "
    "inequivocamente melhor que o baseline, pois trouxe ruído e maior latência. "
    "Os trade-offs foram maior complexidade e latência que o baseline, "
    "além do risco de um orçamento pequeno omitir evidências em perguntas mais "
    "amplas. Neste caso, o contexto final é suficiente porque o Art. 14 e o Art. "
    "15 da página 3 respondem integralmente à carga horária e ao pré-requisito. "
    "A pergunta foi refinada durante os testes exploratórios; outra formulação "
    "chegou a rebaixar o trecho correto e o limite de 200 tokens excluiu a "
    "resposta. O resultado final não demonstra que esse orçamento seja ótimo "
    "ou robusto para outras formulações."
)


## I. Gere o relatório para entrega

A próxima célula reúne identificação, pergunta, resultados, consultas,
contagens, tokens, latências, tabela comparativa, análise e conclusão.

Serão criados dois arquivos em `relatorios/`:

- `relatorio_atividade_rag.md`, fácil de editar;
- `relatorio_atividade_rag.html`, pronto para envio ou para abrir no navegador
  e converter em PDF por **Imprimir → Salvar como PDF**.


In [33]:
from relatorio_atividade import gerar_relatorio

arquivo_md, arquivo_html = gerar_relatorio(
    identificacao=IDENTIFICACAO,
    pergunta=PERGUNTA_ATIVIDADE,
    escopo=ESCOPO_ATIVIDADE,
    justificativa_escopo=JUSTIFICATIVA_ESCOPO,
    alteracao_descricao=ALTERACAO_DESCRICAO,
    avaliacao_manual=AVALIACAO_MANUAL,
    respostas_finais=RESPOSTAS_FINAIS,
    conclusao=CONCLUSAO,
    resultado_baseline=resultado_baseline_atividade,
    tokens_baseline=estimar_tokens_resultados(
        resultado_baseline_atividade
    ),
    etapas_avancado=etapas_atividade,
    saida_avancada=saida_avancada_atividade,
    etapas_alterado=etapas_alteradas,
    saida_alterada=saida_alterada_atividade,
    consultas=CONSULTAS_UNICAS,
    transformacoes_consulta=TRANSFORMACOES_CONSULTA,
)

print(f"Relatório Markdown: {arquivo_md}")
print(f"Relatório HTML: {arquivo_html}")
print("Para PDF: abra o HTML e use Imprimir → Salvar como PDF.")


Relatório Markdown: /home/ubuntu/mba-genai/CONT/aula04/relatorios/relatorio_atividade_rag.md
Relatório HTML: /home/ubuntu/mba-genai/CONT/aula04/relatorios/relatorio_atividade_rag.html
Para PDF: abra o HTML e use Imprimir → Salvar como PDF.


### Checklist antes de enviar

- mesma pergunta nas três configurações;
- apenas uma alteração experimental por comparação;
- consultas geradas registradas;
- resultados ligados a arquivo e página;
- latência e tokens comparados com as mesmas condições;
- respostas qualitativas justificadas com evidências;
- conclusão discute trade-offs, não apenas a melhor métrica;
- HTML ou PDF aberto e revisado antes do envio.


## Opcional: modo interativo original

Execute esta célula somente se quiser usar o menu da professora. Digite `sair`
para encerrar. Ela contém a chamada original que foi separada da definição de
`main()`.


In [ ]:
if __name__ == "__main__":
    main()
